# Code embeddings for the hybrid model (Phase 1: newer models)

This notebook does **no training**. It shows each of the 249 distinct DevMap kernels to several pretrained code models and saves the numbers each model uses to describe the code (its "embedding"). Only the code goes in, never the job numbers.

Models (each saved as its own file, so finished ones are skipped if you run it again):
- `graphcodebert`, `codeberta`: the models from the main study, reading only the first 512 tokens (already done if you ran this before)
- `graphcodebert-pieces`: GraphCodeBERT reading the **whole** kernel in pieces of 512 tokens
- `unixcoder`, `codet5p`, `codesage`, `jina-code`: newer models trained on C code; they also read whole kernels

**How to run:**
1. **Runtime → Change runtime type → T4 GPU** if you have GPU time left (about 5 minutes), otherwise **CPU** (about 30–45 minutes).
2. **Runtime → Run all.** The first cell installs an older version of the `transformers` library that the newer models need, then Colab **restarts by itself** (you may see a "session crashed" or "restarting" message; that is expected).
3. When it has reconnected, press **Runtime → Run all** again. This time it goes all the way through.
4. When Google asks for Drive access, choose your account and tick **Select all**.

If one model fails to load, the notebook prints the error and carries on with the others. Send Claude a screenshot of any error.

The files are saved in your Google Drive under `devmap_results/embeddings`.

In [ ]:
import os, shutil, subprocess

# The newer code models (codet5p, codesage, jina-code) ship their own code written
# for transformers version 4; version 5 removed functions they use. Install the last
# version 4 once, then Colab restarts by itself and you press Run all again.
import transformers
if int(transformers.__version__.split(".")[0]) >= 5:
    print("transformers", transformers.__version__, "found; installing version 4 for the newer models ...")
    subprocess.run(["pip", "install", "-q", "transformers>=4.44,<5"], check=True)
    print("Installed. Colab will now restart by itself.")
    print(">>> When it has reconnected, press Runtime > Run all again. <<<")
    os.kill(os.getpid(), 9)
print("transformers version:", transformers.__version__)

import torch
print("GPU available:", torch.cuda.is_available())
if not os.path.exists("GM_SOURCE_CODE"):
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/marco-siino/GM_SOURCE_CODE.git"], check=True)
os.makedirs("data", exist_ok=True)
shutil.copy("GM_SOURCE_CODE/AMD_DS/dataset-devmap-amd.csv", "data")
shutil.copy("GM_SOURCE_CODE/NVIDIA_DS/dataset-devmap-nvidia.csv", "data")
print("Dataset files:", sorted(os.listdir("data")))

## Helper code (no need to read or change)

In [ ]:
%%writefile devmap.py
"""
Shared helpers: load the DevMap dataset and build the four "exams".

Words used here (same as in our chats):
  row      = one line of the dataset (code + two job numbers + answer)
  program  = one piece of OpenCL code (a "kernel"); several rows can share it
  study    = the rows a model learns from   (training set)
  exam     = the rows it is scored on       (test set)

  application = the benchmark a program was taken from (e.g. "npb-3.3-BT");
                one application contains several programs. 71 in total.
  collection  = the benchmark suite an application belongs to (NPB, Rodinia,
                ...). 7 in total.

The four exams, from easiest to hardest:
  old             - the base paper's exam: shuffle the ROWS, cut into 5 piles.
                    Rows of the same program can land on both sides.
  unseen_program  - shuffle the PROGRAMS, cut into 5 piles. Every row of a
                    program is on one side only.
  unseen_app      - shuffle the APPLICATIONS, cut into 5 piles. Needed because
                    programs of one application share the same job numbers,
                    so those numbers give the application away.
  unseen_suite    - hold out one whole collection at a time, 7 rounds.
"""
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "data")
FILES = {"amd": "dataset-devmap-amd.csv", "nvidia": "dataset-devmap-nvidia.csv"}

# The hand-counted features that ship with the dataset (Grewe et al.):
#   comp      = number of compute operations in the code
#   rational  = number of relational (comparison) operations
#   mem       = number of memory accesses
#   localmem  = number of accesses to local memory
#   coalesced = number of memory accesses that are neatly side by side
#   atomic    = number of atomic operations (always 0 here)
#   transfer  = how much data is sent to the device  (job number 1)
#   wgsize    = how the work is batched               (job number 2)
STATIC_FEATURES = ["comp", "rational", "mem", "localmem", "coalesced", "atomic"]
JOB_FEATURES = ["transfer", "wgsize"]

SUITES = ["amd-sdk", "npb", "nvidia-sdk", "parboil", "polybench", "rodinia", "shoc"]


def suite_of(benchmark_name):
    """Which collection a program comes from, read off its name."""
    b = benchmark_name.lower()
    if b.startswith("amd"):
        return "amd-sdk"
    if b.startswith("npb"):
        return "npb"
    if b.startswith("nvidia"):
        return "nvidia-sdk"
    if b.startswith("parboil"):
        return "parboil"
    if b.startswith("polybench"):
        return "polybench"
    if b.startswith("rodinia"):
        return "rodinia"
    if b.startswith("shoc"):
        return "shoc"
    raise ValueError("unknown collection for " + benchmark_name)


def app_of(benchmark_name):
    """'npb-3.3-BT-exact_rhs1' -> 'npb-3.3-BT' (collection, version, application)."""
    m = re.match(r"^(.*?-\d+(?:\.\d+)*)-([^-]+)-(.*)$", benchmark_name)
    if not m:
        raise ValueError("cannot read application from " + benchmark_name)
    return m.group(1) + "-" + m.group(2)


def code_key(src):
    """Two rows are 'the same program' when their code text is identical
    once extra spaces and line breaks are ignored."""
    return re.sub(r"\s+", " ", str(src)).strip()


def load(machine, data_dir=DATA_DIR):
    """Load one answer sheet ('amd' or 'nvidia') with a few helper columns."""
    df = pd.read_csv(os.path.join(data_dir, FILES[machine]), sep=",")
    df = df.loc[:, [c for c in df.columns if not c.startswith("Unnamed")]].copy()
    df["row_id"] = np.arange(len(df))           # stable id = original file order
    df["suite"] = df["benchmark"].map(suite_of)
    df["app"] = df["benchmark"].map(app_of)
    keys = df["src"].map(code_key)
    df["program_id"] = pd.factorize(keys)[0]     # same code text -> same id
    df["label"] = (df["oracle"] == "GPU").astype(int)   # 0 = CPU, 1 = GPU
    return df


# ----------------------------------------------------------------------------
# The four exams. Each returns a list of rounds; a round is a dict with
# 'name', 'study' (row_ids) and 'exam' (row_ids).
# ----------------------------------------------------------------------------
def old_exam(df):
    """Exactly what the base paper's notebooks do:
    df.sample(frac=1, random_state=1) and then KFold(5, shuffle=False)."""
    shuffled = df.sample(frac=1, random_state=1).reset_index(drop=True)
    rounds = []
    for i, (tr, te) in enumerate(KFold(n_splits=5, shuffle=False).split(shuffled)):
        rounds.append({
            "name": f"pile{i + 1}",
            "study": shuffled.iloc[tr]["row_id"].tolist(),
            "exam": shuffled.iloc[te]["row_id"].tolist(),
        })
    return rounds


def group_piles(groups, labels, n_piles=5, seed=1):
    """Deal whole groups (programs or applications) into n_piles piles.

    Our own version instead of scikit-learn's StratifiedGroupKFold, because
    that one gives different piles in different library versions (Colab's
    1.6 put 364 rows in one AMD pile and 15 in another). This uses only
    numpy's fixed random generator, so the piles are the same everywhere.

    How it deals: shuffle the groups (seed), then take them biggest first and
    put each one in the pile where it keeps the piles most even, both in
    size and in their share of GPU answers. Returns one pile number per row."""
    groups = np.asarray(groups)
    labels = np.asarray(labels)
    names, inverse = np.unique(groups, return_inverse=True)
    sizes = np.bincount(inverse)
    gpus = np.bincount(inverse, weights=labels)
    order = np.random.RandomState(seed).permutation(len(names))
    order = order[np.argsort(-sizes[order], kind="stable")]       # biggest first, ties in shuffled order
    target_rows = len(groups) / n_piles
    target_gpu = max(labels.sum() / n_piles, 1.0)
    pile_rows = np.zeros(n_piles)
    pile_gpu = np.zeros(n_piles)
    pile_of_group = np.empty(len(names), dtype=int)
    for g in order:
        costs = []
        for p in range(n_piles):
            rows, gpu = pile_rows.copy(), pile_gpu.copy()
            rows[p] += sizes[g]
            gpu[p] += gpus[g]
            costs.append((((rows - target_rows) / target_rows) ** 2).sum()
                         + (((gpu - target_gpu) / target_gpu) ** 2).sum())
        best = int(np.argmin(costs))                               # ties -> lowest pile number
        pile_of_group[g] = best
        pile_rows[best] += sizes[g]
        pile_gpu[best] += gpus[g]
    return pile_of_group[inverse]


def fair_exam_unseen_programs(df, seed=1):
    """5 rounds; all rows of a program stay on the same side. Piles are also
    kept roughly balanced in size and CPU/GPU answers."""
    pile = group_piles(df["program_id"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": df.iloc[tr]["row_id"].tolist(),
            "exam": df.iloc[te]["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_apps(df, seed=1):
    """5 rounds; all programs of an application stay on the same side. If the
    exact same code also exists under another application, those study rows
    are dropped so the exam code is truly unseen."""
    pile = group_piles(df["app"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        exam = df.iloc[te]
        study = df.iloc[tr]
        study = study[~study["program_id"].isin(exam["program_id"])]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_collection(df):
    """7 rounds; each round holds out one whole collection. If a held-out
    program's exact code also exists in another collection, those study rows
    are dropped so the exam code is truly unseen."""
    rounds = []
    for suite in SUITES:
        exam = df[df["suite"] == suite]
        study = df[(df["suite"] != suite) & (~df["program_id"].isin(exam["program_id"]))]
        rounds.append({
            "name": suite,
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def build_exam(df, exam, seed=1):
    if exam == "old":
        return old_exam(df)
    if exam == "unseen_program":
        return fair_exam_unseen_programs(df, seed)
    if exam == "unseen_app":
        return fair_exam_unseen_apps(df, seed)
    if exam == "unseen_suite":
        return fair_exam_unseen_collection(df)
    raise ValueError(exam)


EXAMS = ["old", "unseen_program", "unseen_app", "unseen_suite"]
RESHUFFLED_EXAMS = ["unseen_program", "unseen_app"]   # these change with the shuffle seed


def leak_report(df, rounds):
    """For each round: how many exam rows have code that is also in the
    study rows (seen) vs. not (new)."""
    pid = df.set_index("row_id")["program_id"]
    out = []
    for r in rounds:
        study_programs = set(pid.loc[r["study"]])
        seen = sum(1 for x in r["exam"] if pid.loc[x] in study_programs)
        out.append({"round": r["name"], "study_rows": len(r["study"]),
                    "exam_rows": len(r["exam"]), "exam_seen_code": seen,
                    "exam_new_code": len(r["exam"]) - seen})
    return pd.DataFrame(out)

In [ ]:
%%writefile embed_lib.py
"""
Turns each distinct piece of DevMap code into a list of numbers (an
"embedding") using a pretrained model, WITHOUT any training. These numbers
are the model's own summary of what the code looks like and does.

Only the code text goes in - never the job numbers - so the embedding cannot
act as a name tag for the job size.

  embed(...)            runs a model over the 249 distinct kernels, saves an .npz
  load_embeddings(path) reads it back as {code id: vector}
  code_ids(df)          the code id of every dataset row (to look vectors up)

A code id is the MD5 of the whitespace-normalised code (devmap.code_key), so
the same code always gets the same id on any computer.

Two ways of reading a kernel:
  cut      - the first max_len tokens only (what the fine-tuned models do;
             87 of the 249 kernels are longer than 512 tokens and get cut)
  pieces   - the whole kernel, split into pieces of at most max_len tokens;
             the kernel's vector is the average of the pieces' vectors,
             weighted by how many tokens each piece has
"""
import hashlib
import os

import numpy as np
import torch

import devmap

# Each model: where to download it, how many tokens one piece may have, how
# to read long kernels, and model-specific loading details from its model card.
SPECS = {
    # the two models fine-tuned in the main study (first 512 tokens only)
    "codeberta": {"hub": "huggingface/CodeBERTa-language-id", "max_len": 512, "pieces": False},
    "graphcodebert": {"hub": "microsoft/graphcodebert-base", "max_len": 512, "pieces": False},
    # same model, but reading the whole kernel in pieces
    "graphcodebert-pieces": {"hub": "microsoft/graphcodebert-base", "max_len": 512, "pieces": True},
    # newer models (Phase 1)
    "unixcoder": {"hub": "microsoft/unixcoder-base", "max_len": 1024, "pieces": True},
    "codet5p": {"hub": "Salesforce/codet5p-110m-embedding", "max_len": 512, "pieces": True,
                "remote": True},
    "codesage": {"hub": "codesage/codesage-base-v2", "max_len": 1024, "pieces": True,
                 "remote": True, "tokenizer_args": {"add_eos_token": True}},
    "jina-code": {"hub": "jinaai/jina-embeddings-v2-base-code", "max_len": 4096, "pieces": True,
                  "remote": True},
    "codebert": {"hub": "microsoft/codebert-base", "max_len": 512, "pieces": False},
}
MODELS = {k: v["hub"] for k, v in SPECS.items()}       # kept for older scripts


def code_id(src):
    return hashlib.md5(devmap.code_key(src).encode("utf-8")).hexdigest()


def code_ids(df):
    return df["src"].map(code_id).to_numpy()


def unique_codes(data_dir=devmap.DATA_DIR):
    """(ids, texts) for every distinct kernel. The code is the same on both
    machines' answer sheets, so the AMD file is enough."""
    df = devmap.load("amd", data_dir)
    seen, ids, texts = set(), [], []
    for src in df["src"]:
        i = code_id(src)
        if i not in seen:
            seen.add(i)
            ids.append(i)
            texts.append(devmap.code_key(src))
    return ids, texts


def compat_shims():
    """Some models ship their own code written for older versions of the
    transformers library, which import helpers that later versions moved or
    removed. Put those helpers back where the old code looks for them. This only
    makes the old code importable; it does not change any computation."""
    import transformers.modeling_utils as mu
    import transformers.pytorch_utils as pu
    if not hasattr(mu, "Conv1D") and hasattr(pu, "Conv1D"):          # CodeSage
        mu.Conv1D = pu.Conv1D
    if not hasattr(pu, "find_pruneable_heads_and_indices"):          # jina-code on transformers 5
        def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
            mask = torch.ones(n_heads, head_size)
            heads = set(heads) - already_pruned_heads
            for head in heads:
                head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
                mask[head] = 0
            mask = mask.view(-1).contiguous().eq(1)
            index = torch.arange(len(mask))[mask].long()
            return heads, index
        pu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices


def n_tokens(tokenizer, text):
    return len(tokenizer([text], truncation=False)["input_ids"][0])


def split_into_pieces(text, tokenizer, max_len, step=24):
    """Split code into consecutive pieces of at most max_len tokens (special
    tokens included), cutting only between words. Every word lands in exactly
    one piece, in order."""
    words = text.split(" ")
    pieces, current = [], []
    for i in range(0, len(words), step):
        block = words[i:i + step]
        trial = current + block
        if current and n_tokens(tokenizer, " ".join(trial)) > max_len:
            pieces.append(" ".join(current))
            current = list(block)
        else:
            current = trial
    if current:
        pieces.append(" ".join(current))
    return pieces


def _vectors(model, tokenizer, text, max_len, device):
    """(mean vector, first-token vector, tokens used) for one piece of text."""
    enc = tokenizer([text], truncation=True, max_length=max_len, return_tensors="pt")
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device) if "attention_mask" in enc else torch.ones_like(ids)
    try:
        out = model(input_ids=ids, attention_mask=mask)
    except TypeError:                                   # some remote models take ids only
        out = model(ids)
    first = out[0] if not torch.is_tensor(out) else out
    first = first.float()
    if first.dim() == 3:                                # token vectors -> average over real tokens
        m = mask.unsqueeze(-1).float()
        mean = (first * m).sum(1) / m.sum(1).clamp(min=1)
        cls = first[:, 0]
    else:                                               # the model already returns one vector
        mean = cls = first.reshape(1, -1)
    return mean[0].cpu().numpy(), cls[0].cpu().numpy(), int(mask.sum().item())


def embed(model_key, data_dir, out_dir, tokenizer=None, make_encoder=None, device=None,
          max_len=None, pieces=None, log=print):
    """Saves <out_dir>/embeddings__<model_key>.npz with, per kernel:
       mean   - average of the model's token vectors (main choice)
       cls    - the first token's vector
       tokens - tokens actually read;  n_pieces - pieces read
    Skips the work if the file already exists."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"embeddings__{model_key}.npz")
    if os.path.exists(path):
        log(f"{path} already exists, skipping")
        return path
    spec = SPECS.get(model_key, {})
    max_len = max_len or spec.get("max_len", 512)
    pieces = spec.get("pieces", False) if pieces is None else pieces
    if tokenizer is None or make_encoder is None:
        from transformers import AutoModel, AutoTokenizer
        compat_shims()
        hub, remote = spec["hub"], spec.get("remote", False)
        tokenizer = AutoTokenizer.from_pretrained(hub, trust_remote_code=remote, **spec.get("tokenizer_args", {}))
        make_encoder = lambda: AutoModel.from_pretrained(hub, trust_remote_code=remote)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = make_encoder().to(device).eval()

    ids, texts = unique_codes(data_dir)
    means, clss, tokens, n_pieces = [], [], [], []
    with torch.no_grad():
        for k, text in enumerate(texts):
            # On a small GPU a very long piece may not fit in memory. Then read that
            # kernel again in smaller pieces (halving until it fits), rather than crash.
            limit = max_len
            while True:
                try:
                    parts = split_into_pieces(text, tokenizer, limit) if (pieces or limit < max_len) else [text]
                    vecs = [_vectors(model, tokenizer, p, limit, device) for p in parts]
                    break
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if limit <= 128:
                        raise
                    limit //= 2
                    log(f"   kernel {k + 1}: GPU memory full, reading it in pieces of {limit} tokens instead")
            w = np.array([v[2] for v in vecs], dtype=float)
            means.append(np.average(np.stack([v[0] for v in vecs]), axis=0, weights=w))
            clss.append(vecs[0][1])
            tokens.append(int(w.sum()))
            n_pieces.append(len(parts))
            if (k + 1) % 25 == 0 or k + 1 == len(texts):
                log(f"   {model_key}: {k + 1}/{len(texts)} kernels")
    tokens, n_pieces = np.array(tokens), np.array(n_pieces)
    np.savez_compressed(path, ids=np.array(ids), mean=np.stack(means), cls=np.stack(clss),
                        tokens=tokens, n_pieces=n_pieces, model=spec.get("hub", model_key))
    log(f"saved {path}: {len(ids)} kernels, vectors of {len(means[0])} numbers, "
        f"{int((n_pieces > 1).sum())} kernels read in more than one piece")
    return path


def load_embeddings(path, pooling="mean"):
    z = np.load(path, allow_pickle=False)
    return dict(zip(z["ids"].tolist(), z[pooling]))

## Settings

In [ ]:
import os

MODELS = ["graphcodebert", "codeberta", "graphcodebert-pieces",
          "unixcoder", "codet5p", "codesage", "jina-code"]
SAVE_TO_GOOGLE_DRIVE = True

OUT_DIR = "embeddings"
try:
    from google.colab import drive
    if SAVE_TO_GOOGLE_DRIVE:
        drive.mount("/content/drive")            # a Google permission window opens: tick "Select all"
        OUT_DIR = "/content/drive/MyDrive/devmap_results/embeddings"
except ImportError:
    pass
os.makedirs(OUT_DIR, exist_ok=True)
print("Embeddings will be saved in:", OUT_DIR)

## Extract

In [ ]:
import gc, traceback
import torch
import embed_lib

failed = []
for model in MODELS:
    print(f"\n=== {model} ===")
    try:
        embed_lib.embed(model, "data", OUT_DIR)
    except Exception as err:
        failed.append(model)
        print(f"!!! {model} failed: {type(err).__name__}: {err}")
        traceback.print_exc(limit=2)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("\nSaved files:", sorted(os.listdir(OUT_DIR)))
print("Failed models:", failed or "none")

## Download

The files are already in your Google Drive. This also zips and downloads them; send the zip (or the whole `devmap_results` folder) to Claude.

In [ ]:
import shutil
shutil.make_archive("embeddings", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("embeddings.zip")
except ImportError:
    print("Saved embeddings.zip in the current folder.")